GESTIONE MODELLI CON MLFLOW MODEL REGISTRY

MLFlow Model Registry è l'anagrafica delle versioni dei modelli AI. 
Il modello, per esempio Classificatore_mail, ha una versione 7 e una versione 8, puoi indicare quale deve caricare l'applicazione.

Tracking e Registry
- MLflow Tracking registra le prove di addestramento. Ogni esecuzione, detta run, può contenere parametri, metriche come l'F1 e i file del modello.
- Modle Registry: organizza i modelli che scegli di registrare. Se registri un nuovo modlelo addestrato sotto lo stesso nome, MLflow crea una versione e conserva il collegamento alla run di origirne. 

Esemipio con modello Classificatore_mail
Supponiamo che la versione 7 sia in uso, e che tu abbiamo addestrato la 8. Un alias è un nome che punta a una versione.
Scegliamo candidate per il modello in prova e champion per quello da usare in produzione.
- Metti la versione 8 in collaudo: candidate=8, champion=7
- Addestri la versione 8: candidate=8, champion=8
- Decidi di tornare alla 7: candidate=8, champion=7
Qui entra in gioco CI/CD. Puoi far provare alle pipeline la versione esatta models:/classificatore_mail/8 su un insieme fisso di mail già etichettate. 
Dopo il collaudo e l'approvazione, sposti champion sulla versione 8

Dallo stato di 'esperimento' allo stato di 'servizio'

Quando un esperimento diventa ufficialmente un servizio.
Un file di pesi salvato su un server non è un modello pronto per la  produzione. Il Model Registry trasforma un artefatto in un oggetto logico dotato di identità, versione e stato operativo. E' come passare da un bozzetto a matita ad un'opera catalogata.
MLFlow permette di catalogare diverse iterazione dello stesso modello, assegnando loro etichette che ne definiscono l'affidabilità e la destinazione d'uso.

Ma come è strutturato internamente questo archivio Intelligente.

Architettura del REgistry
Centralizzazione e Versionamento
- Registered Model: l'entità di alto livello che raggruppa tutte le versioni di una specifica soluzione AI (es. 'sentiment-analiyzer'). Ho tutte le versioni, versione 1, 2, ecc. ogni versione è immutaibile.
- Model Version: Ogni nuova istanza prodotta da una run di training riceve un numero di versione incrementale e immutabile
- Stage Transition: il processo formale di promozione o retrocessione di una versione attraverso vari stadi di maturità
- Il Registry può essere visto come l'unione ordinata di tutte le versione storiche prodotte per un determinato task.

Non perdiamo mai nulla ma sappiamo chi è il leader attuale della squadra.

3 Stadi Stanrdard
- Staging: è la nostra sala prove, il modello riceve dati reali ma le sue risposte non finisco all'utente finale, è fondamentale per il test
- Production: è il palcoscenico, qui il modello risponde all API ed è sotto i riflettori. Solo un campione alla volta detiene questo titolo.
- Archived: la nostra biblioteca storica, non usiamo più questo modelli ma li conserviamo o per motivi legali o per capire, tra un anno, perchè avevamo preso certe decisioni.

Ma come facciamo a capire chi viene promosso senza basarci solo sull'istinto?
Automazione della Promozione
Webhooks e CI/CD
Il passaggio da Staging a Production non dovrebbe essere manuale, ma automatizzata tramite un CI/CD. MLflow permette di innescare script di validazione che verificano le metriche prima di autorizzare il cambio di stato.
Questo toglie il peso dell'errore umano
La probabilità di successo in produzione è direttamente correlata al rigore di test eseguiti nello stadio di staging.

Tuttavia anche con i migliori test il mondo reale può essere crudele
Cosa succede quando qualcosa va storto?

Affidabilità e Rollback
Garantire la continuità del servizio
Anche il miglior modello può fallire se messo a confronto con i dati reali imprevisti (magari dato da dati degli utenti cambiati). La velocità di rispristino è più importante della perfezione del rilascio.
Il Model Registry facilita il Rollback immediato, permettendo di rimettere in linea una versione stabile precedente in pochi secondi.

Meccanismo di Ripristino.
Gestione delle crisi
- Il segreto è l'Atomic Switching: cioè la capacità di puntare l'alias 'Production' a una versione differente senza tempi di inattività (downtime). Versione precedente che sapevamo essere stabile.
- Version History: accesso immediato a tutti gli artefatti delle versioni precedenti, inclusi i requisiti di sistema e le dipendenze. Accesso non solo ai pesi ma anche a tutto l'ambiente software che quel modello puntava.
- Consistency Check: verifica che la versione su cui si sta effettuando il roolback sia compatibile con l'infrastruttura attuale.
- Il tempo di ripristino (MTTR) è minimizzato quando il sistema di registry permette il cambio di puntatore senza ricaricamento fisico dei dati.

Preventizione dei Fallimenti.
- Shadow Deploy: tecnica in cui la versione v2 gira in parallelo alla v1 in produzione: riceve lo stesso traffico ma le sue risposte non vengono mostrate all'utente. Se notiamo anomalie sulla v2 non la candidiamo mai.
- Canary Release: distribuzione graduale della nuova versione del registry a una piccola percentuale di utenti (es. 5% degli utenti) per monitorare eventuali anomalie.
- Performance Comparison: utilizzo delle dashboard di MLflow per confrontare la 'drift' tra la versione in roolback e quella fallita per capire la causa dell'errore.

Per implementare tutto questo servono dei cancelli di sicurezza che vengono automatizzati

Quality Gates
Validazione automatica.
Il registry funge da guardiano (gatekeeper). Un modello non può essere promosso se non supera una funzione indicatrice basata su soglie di performance.
L'autorizzazione al rilascio è concessa solo se le prestazioni correnti superano la soglia critica stabilita dal team di ML Engineering.
I Quality Gates devono essere definiti in modo che controllino non solo l'accuratezza ma anche la letenza. Se il modello è preciso ma lento per il nostro server, il gate lo blocca in staging.

Ma un modello non è solo performance è anche storia e contesto.
Tra un anno nessuno si ricorderà perchè avete scelto certi iperparametri alle 3 di note di un lunedi.
Senza metadati un modello è orfano digitale. Un debito tecnico che esploderà quando dovrete aggiornare il sistema e non si sa da dove iniziare.
A lungo termine, la domanda più difficile non è 'come funzione' ma 'chi lo ha addestrato e su quali dati?'. La documentazione integrata è vitralel per la manutenzione.
Dobbiamo arrichire le versioni nel registry con metadati descrittivi, link ai dataset originali e note degli sviluppatori.

Come connettere questi punti per avere una tracciabilità totale.

Tracciabilità e Lineage
Il Lineage è come un filo rosso che connette tutti questi punti.
- Model Description: documentazione Markdown interna al registry che spiega il caso d'uso e i limiti del modello
- Dataset Linking: associazione univoca tra la versione del modello e l'hash del dataest utilizzato (provenienze da DVC o database)
- Provenance Tracking: registrazione automatica dell'utente che ha creato la run e della macchina utilizzata
- Ogni ID di modello deve essere mappabile a un set completo di informazioni sul dataset, sui parametri e sulle metriche ottenute.
In caso di audit o problemi legali, questo è la nostra assicurazione sulla vita professionale.

Questa trasparenza facilita anche la collaborazione all'interno del team.

Collaborazione e Governance
- Access Control (RBAC): gestione dei permessi, decidere chi può registrare nuove versioni e chi ha l'autorità di promuovere un modello allo stadio di 'Production'
- Audit log: tracciamento di tutti i cambiamenti di stage. Utile per capire chi ha autorizzato un deploy che ha poi richiesto un rollback
- Model Cards: creazione automatizzate di schede tecniche che riassumono l'etica del modello, i bias rilevati e le performance per diversi sottogruppi di dati.
Tutto questo non è burocrazia ma sicurezza.
E' il modo in cui comunichiamo la responsabilità sociale della nostra AI

L'importanza del Context
Evitare modello 'orfani'
Un modello senza metadati è un debito tecnico latente. Senza il contesto del training, è impossibile aggiornarlo o correggerlo se le performance calano.
La stabilità di un'azienda basata sull'AI dipende dalla capacità di mantenere un archivio storico navigabile e coerente di tutta la propria proprietà intelletuale algoritmica.









